<a href="https://colab.research.google.com/github/timraiswell/ai-engineer/blob/main/02-evals-basics/01-measuring-outputs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evals I: Measuring a Probabilistic System

**Goal:** Install the habit that separates an AI engineer from a prompt tinkerer: putting a *number* on "is it good," on the simplest possible system, before you build anything you'd need to tune.

> **🔵 Where this fits —** evals are the spine of this course, so they come in **two passes.** This is the light first pass: the golden-set → metric → measured-change loop on a tiny task, placed *early* so the habit is in hand before you build anything. **Section 04 (Evals II)** is the rigorous version on the real RAG system, with LLM-as-judge, human calibration, and regression-as-CI. Between them, evals recur in every section (RAG retrieval, agent trajectories, fine-tuning decisions, production monitoring). Short by design; the depth is in section 04.

Part of [ai-engineer-notebooks](https://github.com/calmrocks/ai-engineer-notebooks), the hands-on companion to the [FDE / AI Engineer transition plan](https://www.calm.rocks/resources/career-development/transition-fde-ai-engineer/).

## Setup

Each notebook is self-contained, so the next two cells stand it up from scratch:

1. **Install dependencies.** The `aien` package (this repo) carries the shared setup helper and pulls in the `groq` client, the only dependency this notebook needs.
2. **Load your API key.** Free key at [console.groq.com](https://console.groq.com/) (no credit card); in Colab add it via the **key icon** → **Add new secret** named exactly `GROQ_API_KEY`, notebook access on. Locally, set `GROQ_API_KEY` in your environment.

(Full walkthrough: [00-setup/00-environment.ipynb](https://colab.research.google.com/github/calmrocks/ai-engineer-notebooks/blob/main/00-setup/00-environment.ipynb).)

In [1]:
%pip install -q "git+https://github.com/calmrocks/ai-engineer-notebooks.git"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.7 MB/s eta 0:00:00


In [2]:
from aien import setup

# Loads GROQ_API_KEY and returns a ready Groq client.
client, MODEL = setup()

Groq client ready. MODEL = openai/gpt-oss-120b


> **🔵 Hitting a rate limit? Switch models.** Eval notebooks are token-heavy (one model call per case, run repeatedly), and Groq's free tier caps both tokens-per-minute and tokens-per-day **per model**. If you get a `429`, point `setup()` at a different model; it has its own separate budget:
> ```python
> client, MODEL = setup(model='openai/gpt-oss-20b')     # or 'qwen/qwen3.6-27b'
> ```
> Each free model has its own limits; pick one with headroom from the [Groq rate-limits page](https://console.groq.com/docs/rate-limits). For a *verification* pass any capable chat model is fine. The methodology is the point, not the exact scores (the "develop on cheap, eval on target" rule from section 00). Avoid the agentic `groq/compound*` models here: they browse and run code, which would defeat the grounded-answer lessons.

## Why this comes before everything else

You just learned to get structured output and call tools (section 01). The instinct now is to build something bigger: RAG, an agent. **Resist it for one notebook.** Here's why.

In normal software, "does it work?" has a deterministic answer: the test passes or it doesn't, the compiler complains or it doesn't. LLM systems break that. The same prompt can give a good answer now and a wrong one on the next call; a change that helps one input silently breaks three others; "it looked right when I tried it" is how broken systems ship. You cannot *eyeball* an LLM system into working.

The only thing that lets you build, tune, or change anything with confidence is a **number that says whether it's getting better or worse.** That number is an **eval**, the discipline the plan calls *"the differentiator."* We install it now, on the tiny extraction task from section 01, where the mechanics are obvious, so it's second nature by the time you're tuning retrieval and agents where eyeballing fails hardest.

## The system under test: the section-01 extraction task

A compact version of the support-ticket extractor from `01-model-apis/01-structured-output`: free-text email in, structured ticket out. This is our system-under-test, small enough that what a *good* output looks like is unambiguous, which is exactly what you want when you're learning to measure.

In [3]:
import json, re
from groq import BadRequestError

TICKET_TOOL = {
    "type": "function",
    "function": {
        "name": "record_ticket",
        "description": "Record a structured support ticket from a customer email.",
        "parameters": {
            "type": "object",
            "properties": {
                "summary":  {"type": "string"},
                "priority": {"type": "string", "enum": ["low", "medium", "high", "urgent"]},
                "category": {"type": "string", "enum": ["billing", "bug", "account", "other"]},
            },
            "required": ["summary", "priority", "category"],
        },
    },
}

def extract_ticket(email, system=None, tries=3):
    # Forcing tool_choice makes the model emit the call as text the API parses; when
    # that parse fails Groq raises 400 tool_use_failed instead of returning a message.
    # It's intermittent, so retry a couple of times before giving up (section 08 goes
    # deep on this reliability pattern).
    messages = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": f"Extract the ticket:\n\n{email}"}]
    for _ in range(tries):
        try:
            resp = client.chat.completions.create(
                model=MODEL, max_tokens=200, tools=[TICKET_TOOL],
                tool_choice={"type": "function", "function": {"name": "record_ticket"}},
                messages=messages,
            )
            return json.loads(resp.choices[0].message.tool_calls[0].function.arguments)
        except BadRequestError as e:
            if 'tool_use_failed' in str(e):
                continue
            raise
    return {"summary": "", "priority": None, "category": None}  # sentinel: counts as a failed case

# Stable private alias so the v2 variant below can call the base extractor even after
# we swap the `extract_ticket` name (keeps a re-run of that cell from recursing).
_extract_base = extract_ticket

print(extract_ticket("Hi, I was charged twice for my subscription this month. Please refund. Urgent!"))

{'category': 'billing', 'priority': 'urgent', 'summary': 'Charged twice for subscription this month; request refund'}


## Step 1: a golden set, the thing you measure against

An eval needs *ground truth*: inputs paired with what a correct output should be. That's a **golden set**: hand-written, trusted, small. Ten good cases beat a thousand unlabeled ones. Cover the easy path *and* the cases you're worried about (here: an ambiguous priority, a category that could go two ways, a terse email).

In [4]:
# Each case: the input email + the fields a correct extraction MUST get right.
# We don't pin the exact summary wording (many phrasings are fine) -- only the
# machine-checkable fields. Choosing what to assert is half the skill of evals.
GOLDEN = [
    {"email": "I was double-charged for my plan this month, need a refund asap.",
     "priority": "urgent", "category": "billing"},
    {"email": "The export button throws a 500 error every time I click it.",
     "priority": "high", "category": "bug"},
    {"email": "How do I change the email address on my account?",
     "priority": "low", "category": "account"},
    {"email": "Your product is fine I guess. Just leaving feedback.",
     "priority": "low", "category": "other"},
    {"email": "URGENT: production is down, your API returns 503 for all calls!!",
     "priority": "urgent", "category": "bug"},
    {"email": "I'd like to upgrade to the annual plan — what's the price?",
     "priority": "low", "category": "billing"},
]
print(len(GOLDEN), "golden cases")

6 golden cases


## Step 2: metrics — turn right/wrong into a number

For structured output the metrics are simple and deterministic, with no LLM judge needed yet (that's section 04, for when outputs are free text you *can't* string-match). Here:

- **Schema validity** — did it return the right shape at all? (a call that "succeeded" but returned junk is a failure)
- **Field accuracy** — for the fields we pinned, did it get them right?

Run it and read the per-case table *and* the aggregate. The aggregate is what you'll track over time; the per-case rows are where you go to debug.

In [5]:
def valid_schema(t):
    return (isinstance(t.get("summary"), str)
            and t.get("priority") in {"low","medium","high","urgent"}
            and t.get("category") in {"billing","bug","account","other"})

def evaluate(golden):
    rows, correct_fields, total_fields, valid = [], 0, 0, 0
    for case in golden:
        got = extract_ticket(case["email"])
        ok_schema = valid_schema(got)
        valid += ok_schema
        # field accuracy on the pinned fields
        p_ok = got.get("priority") == case["priority"]
        c_ok = got.get("category") == case["category"]
        correct_fields += p_ok + c_ok; total_fields += 2
        rows.append((case["email"][:42], got.get("priority"), p_ok, got.get("category"), c_ok))
        print(f"  pri={str(got.get('priority')):8}{'ok' if p_ok else 'XX'}  "
              f"cat={str(got.get('category')):8}{'ok' if c_ok else 'XX'}  | {case['email'][:40]}")
    print("-"*72)
    print(f"schema valid : {valid}/{len(golden)}")
    print(f"field accuracy: {correct_fields}/{total_fields} = {correct_fields/total_fields:.0%}")
    return correct_fields/total_fields

score = evaluate(GOLDEN)

  pri=urgent  ok  cat=billing ok  | I was double-charged for my plan this mo
  pri=high    ok  cat=bug     ok  | The export button throws a 500 error eve
  pri=medium  XX  cat=account ok  | How do I change the email address on my 
  pri=low     ok  cat=other   ok  | Your product is fine I guess. Just leavi
  pri=urgent  ok  cat=bug     ok  | URGENT: production is down, your API ret
  pri=medium  XX  cat=billing ok  | I'd like to upgrade to the annual plan —
------------------------------------------------------------------------
schema valid : 6/6
field accuracy: 10/12 = 83%


## Step 3: now a change is a decision, not a guess

Here's the whole point. Suppose you want to "improve" the extractor by adding a rule to the prompt. Without an eval, you'd try it, eyeball one output, and ship on vibes. With the eval, you **measure both and compare**: the change either moves the number or it doesn't.

In [6]:
# A prompt tweak: add explicit priority guidance. Does it actually help? MEASURE it.
PRIORITY_GUIDE = ("Priority guide: outages/data-loss/double-charges = urgent; "
                  "broken features = high; questions and feedback = low.")

def extract_ticket_v2(email):
    # Reuses the base extractor (with its retry) via the stable alias, just adds a
    # system prompt — so v2 is genuinely a one-variable change from v1.
    return _extract_base(email, system=PRIORITY_GUIDE)

# Swap the extractor the harness uses, re-run, compare to `score` from Step 2.
globals()["extract_ticket"] = extract_ticket_v2
print("--- v2 (with priority guidance):")
score_v2 = evaluate(GOLDEN)
print(f"\nv1 = {score:.0%}  ->  v2 = {score_v2:.0%}  "
      f"({'better' if score_v2>score else 'no better — do not ship the extra tokens'})")

--- v2 (with priority guidance):
  pri=urgent  ok  cat=billing ok  | I was double-charged for my plan this mo
  pri=high    ok  cat=bug     ok  | The export button throws a 500 error eve
  pri=low     ok  cat=account ok  | How do I change the email address on my 
  pri=low     ok  cat=other   ok  | Your product is fine I guess. Just leavi
  pri=urgent  ok  cat=bug     ok  | URGENT: production is down, your API ret
  pri=low     ok  cat=billing ok  | I'd like to upgrade to the annual plan —
------------------------------------------------------------------------
schema valid : 6/6
field accuracy: 12/12 = 100%

v1 = 83%  ->  v2 = 100%  (better)


That comparison, *v1 vs v2, by a number, on the same cases*, is the atom of every improvement you'll make for the rest of this course and your career. You just did, on a toy task, exactly what section 04 does rigorously (golden sets, LLM-as-judge, and wiring evals into CI) for the RAG system, what section 05 does for agent trajectories, and what section 08 feeds back from production traces.

> **⭐ Key takeaway —** before you tune *anything* (a prompt, a chunk size, a retriever, a model) ask "what's my golden set and what's the number?" If you don't have one, you're not improving the system, you're just changing it.

Every section from here on assumes you now think this way.

## Exercises

1. **Break it, then catch it.** Add a deliberately bad rule to the v2 system prompt (e.g. "everything is low priority"). Confirm the eval score *drops*. You've just proven your eval can detect a regression, which is the whole reason it exists.
2. **Add the cases you fear.** Write 4 more golden cases targeting ambiguity: an email that's both billing and bug, a passive-aggressive one, a non-English phrase, an empty-ish message. Which does the extractor get wrong? Each failure is a case worth keeping.
3. **Add a metric.** Extend `evaluate` to also measure summary quality with a cheap proxy: e.g. is the summary between 5 and 100 chars and does it share a keyword with the email? Note where a deterministic proxy is enough and where you'd eventually want an LLM judge (section 04).
4. **Cost of the eval.** The harness makes one model call per case. How would you keep a golden set fast and cheap enough to run on *every* change? (This question is exactly what section 04's regression-in-CI notebook answers.)